# Insurance Portfolio Analytics — Python Assessment
**Müəllif:** Aysu Qadirova

## 1. Data Import və Keyfiyyət Yoxlaması (Data Cleaning)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

xls = r'C:\Users\User\OneDrive\Документы\data-analyst-task-datasets.xlsx'
my_file = pd.ExcelFile(xls)

customers = pd.read_excel(my_file, sheet_name='customers')
policies = pd.read_excel(my_file, sheet_name='policies')
claims = pd.read_excel(my_file, sheet_name='claims')
payments = pd.read_excel(my_file, sheet_name='payments')

# Tarix sütunlarını datetime formatına çeviririk
policies['start_date'] = pd.to_datetime(policies['start_date'])
policies['end_date'] = pd.to_datetime(policies['end_date'])
claims['claim_date'] = pd.to_datetime(claims['claim_date'])
customers['birth_date'] = pd.to_datetime(customers['birth_date'])
customers['join_date'] = pd.to_datetime(customers['join_date'])

# broker_name üçün NaN dəyərləri "Not Broker" ilə əvəz edirik (channel != broker olduqda normaldır)
policies['broker_name'] = policies['broker_name'].fillna('Not Broker')

# Data keyfiyyəti yoxlaması: təkrar sətirlər, sütun tipləri, boş dəyərlər (sütun bazında)
for df, name in [(customers, 'customers'), (policies, 'policies'), (claims, 'claims'), (payments, 'payments')]:
    print(f"\n--- {name.upper()} ---")
    print(f"Təkrar sətir sayı: {df.duplicated().sum()}")
    print("Sütun tipləri:")
    print(df.dtypes)
    null_cols = df.isnull().sum()
    null_cols = null_cols[null_cols > 0]
    if len(null_cols) > 0:
        print("Boş dəyəri olan sütunlar:")
        print(null_cols)
    else:
        print("Boş dəyər yoxdur.")

# Biznes məntiqli əlavə təmizləmə: join_date, birth_date-dən əvvəl ola bilməz
real_customer = len(customers)
customers = customers[customers['join_date'] >= customers['birth_date']].copy()
print(f"\ncustomers: {real_customer - len(customers)} yanlış tarixli müştəri silindi.")

# Polis müddətindən sonra qeydə alınan iddialar (yalnız məlumat üçün, silinmir)
merge_claims = claims.merge(policies[['policy_id', 'end_date']], on='policy_id', how='left')
claims['after_policy'] = merge_claims['claim_date'] > merge_claims['end_date']
print(f"claims: {claims['after_policy'].sum()} ədəd polis müddətindən sonrakı iddia aşkarlandı (analizdən çıxarılmayıb, qeyd olaraq saxlanılıb).")

## 2. Analitik Model — Datasetlərin Birləşdirilməsi (Data Preparation)

In [ ]:
claims_sum = claims.groupby('policy_id').agg(
    count_claims=('claim_id', 'count'),
    claims_amount=('claim_amount', 'sum'),
    claims_approved=('approved_amount', 'sum')
).reset_index()

payments_sum = payments.groupby('policy_id').agg(
    total_payments=('payment_id', 'count'),
    total_paid=('payment_amount', 'sum')
).reset_index()

work_book = (
    policies
    .merge(customers, on='customer_id', how='left')
    .merge(claims_sum, on='policy_id', how='left')
    .merge(payments_sum, on='policy_id', how='left')
)

# fillna yalnız aqreqasiyadan gələn ədədi sütunlara tətbiq olunur (kateqorik sütunlar toxunulmaz qalır)
agg_cols = ['count_claims', 'claims_amount', 'claims_approved', 'total_payments', 'total_paid']
work_book[agg_cols] = work_book[agg_cols].fillna(0)

print(work_book.head())

## 3. Baza KPI-lar

In [ ]:
Ümumi_sığorta = work_book['premium_amount'].sum()
Ümumi_zərər = work_book['claims_amount'].sum()
umumi_tasdiqlenmis_zerer = work_book['claims_approved'].sum()
Zərər_faizlə = (Ümumi_zərər / Ümumi_sığorta) * 100

print(f"Ümumi Sığorta: {Ümumi_sığorta:,.2f} AZN")
print(f"Ümumi Zərər (tələb olunan): {Ümumi_zərər:,.2f} AZN")
print(f"Ümumi Təsdiqlənmiş Zərər Məbləği: {umumi_tasdiqlenmis_zerer:,.2f} AZN")
print(f"Zərər faizi (claim ratio): {Zərər_faizlə:,.2f}%")

Yenilənmiş = (work_book['status'] == 'Renewed').sum()
Müddəti_bitmiş = (work_book['status'] == 'Expired').sum()
Faiz_dərəcəsi = (Yenilənmiş / (Yenilənmiş + Müddəti_bitmiş)) * 100

print(f"Yenilənmə Faizi: {Faiz_dərəcəsi:.2f}%")

## 4. Sığorta Növlərinə Görə Paylanma və Outlier Təhlili (IQR)

In [ ]:
növlərə_görə_məbləğ = work_book.groupby('insurance_type')['premium_amount'].sum()
print(növlərə_görə_məbləğ)

Q1 = work_book['premium_amount'].quantile(0.25)
Q3 = work_book['premium_amount'].quantile(0.75)
IQR = Q3 - Q1

yuxarı_sərhəd = Q3 + (1.5 * IQR)
aşağı_sərhəd = Q1 - (1.5 * IQR)

yuxarı_outliers = work_book[work_book['premium_amount'] > yuxarı_sərhəd]
aşağı_outliers = work_book[work_book['premium_amount'] < aşağı_sərhəd]
outliers = len(yuxarı_outliers) + len(aşağı_outliers)

print(f"Normadan yuxarı limit: {yuxarı_sərhəd:,.2f} AZN")
print(f"Normadan aşağı limit: {aşağı_sərhəd:,.2f} AZN")
print(f"Anomal müqavilələrin sayı: {outliers}")

# Sığorta növünə görə ayrı-ayrı paylanma (tapşırıq "növlərə görə" paylanma istəyir)
plt.figure(figsize=(9, 5))
sns.boxplot(data=work_book, x='insurance_type', y='premium_amount', palette='Set2')
plt.title('Sığorta Növlərinə Görə Premium Paylanması')
plt.xlabel('Sığorta Növü')
plt.ylabel('Premium Amount (AZN)')
plt.show()

## 5. Yüksək Riskli Müştərilərin Erkən Aşkarlanması

In [ ]:
# NOT: Risk həddi (count_claims > 2) Sual 8-dəki seqmentasiya ilə eyni olması üçün
# bilərəkdən uzlaşdırılıb — hər iki yerdə eyni "yüksək risk" tərifi istifadə olunur.
work_book['loss_ratio'] = work_book['claims_amount'] / work_book['premium_amount']
riskli = work_book[(work_book['loss_ratio'] > 0.8) | (work_book['count_claims'] > 2)]

print(f"Yüksək riskli müqavilələrin sayı: {len(riskli)}")

## 6. Regional Risk Mənzərəsi

In [ ]:
region = work_book.groupby('region').agg(
    müqavilə_sayı=('policy_id', 'count'),
    zərər_sayı=('count_claims', 'sum'),
    zərər_məbləği=('claims_amount', 'sum')
)

region['tezlik'] = region['zərər_sayı'] / region['müqavilə_sayı']
region['ağırlıq'] = region['zərər_məbləği'] / region['zərər_sayı']

print(region[['tezlik', 'ağırlıq']])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
region[['tezlik']].plot(kind='bar', ax=axes[0], color='skyblue', legend=False)
axes[0].set_title('Regionlar üzrə Zərər Tezliyi')
region[['ağırlıq']].plot(kind='bar', ax=axes[1], color='salmon', legend=False)
axes[1].set_title('Regionlar üzrə Zərər Ağırlığı (AZN)')
plt.tight_layout()
plt.show()

## 7. Metriklər Arasında Korrelyasiya

In [ ]:
sütunlar = ['premium_amount', 'claims_amount', 'claims_approved', 'total_paid']
korrelyasiya = work_book[sütunlar].corr()

display(korrelyasiya.style.background_gradient(cmap='coolwarm').format("{:.2f}"))

plt.figure(figsize=(7, 5))
sns.heatmap(korrelyasiya, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Rəqəmsal Metriklər Arasında Korrelyasiya Heatmap')
plt.show()

## 8. Müştərilərin Risk Səviyyələrinə Görə Seqmentasiyası

In [ ]:
work_book['frequency'] = work_book['count_claims']

def risk_teyin_et(row):
    if row['loss_ratio'] > 0.8 or row['count_claims'] > 2:
        return 'Yüksək Risk'
    elif row['loss_ratio'] > 0.3 or row['count_claims'] >= 1:
        return 'Orta Risk'
    else:
        return 'Aşağı Risk'

work_book['risk_group'] = work_book.apply(risk_teyin_et, axis=1)

risk_sayi = work_book['risk_group'].value_counts()
print("Üç dərəcəli riskli müştəri bölgüsü:")
print(risk_sayi)

plt.figure(figsize=(6, 6))
risk_sayi.plot(kind='pie', autopct='%1.1f%%', colors=['red', 'orange', 'green'])
plt.title('Müştərilərin Risk Səviyyələrinə Görə Paylanması')
plt.ylabel('')
plt.show()

## Yekun Analitik Xülasə (Executive Summary)

- **Data Keyfiyyəti:** Bütün 4 cədvəldə təkrarlanan sətirlər, boş dəyərlər və sütun tipləri yoxlanılmış; tarix sütunları `datetime` formatına gətirilmiş; məntiqsiz tarixlər (doğum tarixindən əvvəl qoşulma və s.) filtrlənmişdir.
- **Regional Risk:** Zərər tezliyi və ağırlığı üzrə regional fərqliliklər aşkar edilmişdir. Müəyyən regionlarda hadisə sayı az olsa da, tək bir zərərin orta məbləği (ağırlığı) daha yüksəkdir.
- **Risk Seqmentasiyası:** Zərərlilik dərəcəsi (Loss Ratio) > 0.8 və ya hadisə sayı > 2 olan müştərilər ardıcıl olaraq (həm erkən aşkarlama, həm seqmentasiya mərhələsində) "Yüksək Risk" kateqoriyasına aid edilmişdir. Bu seqment şirkətin ümumi zərər portfelinin əsas hissəsini təşkil edir.

### Tövsiyə
Yüksək riskli müştəri seqmenti üçün sığorta haqlarının (premium) artırılması və ya anderraytinq şərtlərinin sərtləşdirilməsi tövsiyə olunur.